# 01 — Casting dataset exploration

**Colab:** Runtime → Change runtime type → **GPU** (optional for this notebook; CPU is fine).

Mount Drive if data lives on Google Drive:
```python
# from google.colab import drive
# drive.mount('/content/drive')
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.data import detect_layout, load_all_samples, resolve_data_root, split_train_val
from src.utils import load_config

In [ ]:
cfg = load_config(ROOT / 'configs' / 'default.yaml')
data_root = resolve_data_root(cfg)
print('Data root:', data_root)
print('Detected layout:', detect_layout(data_root))

train_pool, test_samples, layout = load_all_samples(data_root, layout=cfg['data'].get('layout', 'auto'))
train_samples, val_samples = split_train_val(
    train_pool,
    val_fraction=cfg['data']['val_fraction'],
    seed=cfg['seed'],
    group_augmentations=cfg['data'].get('group_augmentations', True),
)
print(f'layout={layout} | train={len(train_samples)} val={len(val_samples)} test={len(test_samples)}')

In [ ]:
def to_df(samples, split_name):
    return pd.DataFrame({
        'split': split_name,
        'label': ['defective' if s.label else 'acceptable' for s in samples],
        'group_id': [s.group_id for s in samples],
        'path': [str(s.path) for s in samples],
    })

df = pd.concat([
    to_df(train_samples, 'train'),
    to_df(val_samples, 'val'),
    to_df(test_samples, 'test'),
])
df.groupby(['split', 'label']).size().unstack(fill_value=0)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, sample in zip(axes.flatten(), test_samples[:8]):
    ax.imshow(Image.open(sample.path).convert('L'), cmap='gray')
    ax.set_title('defective' if sample.label else 'acceptable', fontsize=9)
    ax.axis('off')
plt.suptitle('Sample test images')
plt.tight_layout()
plt.show()

## Augmentation grouping sanity check

If using the Kaggle augmented split, verify that filename groups (e.g. `cast_ok_3_1`, `cast_ok_3_2`) never appear in both train and val.

In [ ]:
train_groups = set(s.group_id for s in train_samples)
val_groups = set(s.group_id for s in val_samples)
overlap = train_groups & val_groups
print('Overlapping groups between train and val:', len(overlap))
assert len(overlap) == 0, 'Leakage: same augmentation group in train and val'